# Demo G: PagedAttention (vLLM)

**Platform:** Lightning.ai Studio (A100 GPU)

**Goal:** Prove PagedAttention eliminates KV cache fragmentation, allowing 4x more concurrent users.

## Concept
Without PagedAttention: KV cache is allocated in contiguous blocks. Fragmentation wastes 50-70%.
With PagedAttention: page table with small non-contiguous blocks. No waste.

## How to run
1. Open Lightning.ai Studio with A100 GPU
2. Start vLLM server in terminal (see cell below)
3. Run benchmark cells in this notebook

In [ ]:
# Install ALL dependencies first
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch', 'transformers', 'accelerate', 'matplotlib', 'requests', 'tqdm',
                       'numpy<2', 'scipy>=1.14'])

import time, requests, torch
from tqdm import tqdm
import matplotlib.pyplot as plt

MODEL = 'mistralai/Mistral-7B-v0.1'
PORT = 8000
BASE_URL = f'http://localhost:{PORT}/v1'

# ─── SHARED TEST (same for HF and vLLM) ───
N_REQUESTS = 5
N_TOKENS = 10
PROMPTS = [f'Explain machine learning concept {i} in simple terms:' for i in range(N_REQUESTS)]

def send_to_vllm(prompts, max_tokens, label=''):
    """Send prompts to vLLM server, return elapsed time and throughput."""
    t0 = time.perf_counter()
    for prompt in prompts:
        requests.post(f'{BASE_URL}/completions', json={
            'model': MODEL, 'prompt': prompt, 'max_tokens': max_tokens, 'temperature': 0
        })
    elapsed = time.perf_counter() - t0
    throughput = (len(prompts) * max_tokens) / elapsed
    print(f'  [{label}] {len(prompts)} requests, {elapsed:.2f}s, {throughput:.0f} tok/s')
    return elapsed, throughput

print(f'Ready. Test: {N_REQUESTS} requests x {N_TOKENS} tokens.')

## Part 1: HuggingFace Baseline (No PagedAttention)

Sequential generation. No batching. No memory management.
This is what happens without an inference engine.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model
print('Loading Mistral-7B...')
hf_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto', token=False)
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL, token=False)

# Warmup: 3 calls to fully compile CUDA kernels
print('Warming up (3 calls)...')
with torch.no_grad():
    for wi in range(3):
        w = hf_tokenizer(f'Warmup call number {wi}', return_tensors='pt').to('cuda')
        hf_model.generate(**w, max_new_tokens=5, pad_token_id=hf_tokenizer.eos_token_id)
torch.cuda.synchronize()
print('Warmup done.')

# Run SAME test that vLLM will run
print(f'Running {N_REQUESTS} requests x {N_TOKENS} tokens (sequential)...')
torch.cuda.synchronize()
hf_start = time.perf_counter()
for p in tqdm(PROMPTS, desc='HF generating'):
    inp = hf_tokenizer(p, return_tensors='pt').to('cuda')
    with torch.no_grad():
        hf_model.generate(**inp, max_new_tokens=N_TOKENS, do_sample=False, pad_token_id=hf_tokenizer.eos_token_id)
torch.cuda.synchronize()
hf_time = time.perf_counter() - hf_start
hf_throughput = (N_REQUESTS * N_TOKENS) / hf_time

print(f'\nHF Baseline: {hf_time:.1f}s, {hf_throughput:.0f} tok/s')

# Free GPU
del hf_model, hf_tokenizer
torch.cuda.empty_cache()
print('GPU freed. Start vLLM server now.')


## Part 2: vLLM with PagedAttention

Run in terminal:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --gpu-memory-utilization 0.90 \
    --port 8000
```

Wait for `Uvicorn running on http://0.0.0.0:8000`

In [ ]:
# Verify server is running
try:
    resp = requests.get(f'{BASE_URL}/models')
    print(f'Server ready: {[m["id"] for m in resp.json()["data"]]}')
except Exception as e:
    print(f'Server not running. Start it first. Error: {e}')

In [ ]:
# Run SAME test on vLLM
print(f'Running {N_REQUESTS} requests x {N_TOKENS} tokens on vLLM...')
# Run with progress bar
vllm_start = time.perf_counter()
for p in tqdm(PROMPTS, desc='vLLM generating'):
    requests.post(f'{BASE_URL}/completions', json={'model': MODEL, 'prompt': p, 'max_tokens': N_TOKENS, 'temperature': 0})
vllm_time = time.perf_counter() - vllm_start
vllm_throughput = (N_REQUESTS * N_TOKENS) / vllm_time
print(f'  [vLLM] {N_REQUESTS} requests, {vllm_time:.2f}s, {vllm_throughput:.0f} tok/s')

print(f'\nvLLM: {vllm_time:.1f}s, {vllm_throughput:.0f} tok/s')
print(f'Speedup over HF: {vllm_throughput/hf_throughput:.1f}x')

## Comparison: Same Test, Different Engine

In [ ]:
speedup = vllm_throughput / hf_throughput

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    ['HuggingFace\n(no PagedAttention)', 'vLLM\n(PagedAttention)'],
    [hf_throughput, vllm_throughput],
    color=['#ffe4e6', '#dcfce7'], edgecolor='#000', linewidth=1.2
)
ax.text(0, hf_throughput + 5, f'{hf_throughput:.0f} tok/s', ha='center', fontsize=12)
ax.text(1, vllm_throughput + 5, f'{vllm_throughput:.0f} tok/s', ha='center', fontsize=12)
ax.text(1, vllm_throughput * 0.5, f'{speedup:.1f}x', ha='center', fontsize=16, color='#166534', fontweight='bold')
ax.set_ylabel('Throughput (tok/s)')
ax.set_title(f'Same Test: {N_REQUESTS} requests x {N_TOKENS} tokens, same GPU', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Key Takeaway

| | HuggingFace | vLLM (PagedAttention) |
|---|---|---|
| Memory | Contiguous, fragmented | Paged, no waste |
| Batching | Sequential | Continuous |
| 50+ users | OOM | Works |
| Code change | None | `pip install vllm` |

**PagedAttention is vLLM's default. You get it for free.**